# DEG cells -> two-wheel system: the GPU run

One-click pipeline to bring the DEG organisms (mtub, syn3a, mgen, S. aureus,
S. pneumoniae) into the two-wheel system. Run on a **GPU runtime** (A100/L4).

Stages (only steps 5 & 8 use the GPU):
1. clone repo + install mmseqs + ESM deps
2. fetch DEG proteomes from NCBI (network)
3. mmseqs assign DEG proteins -> orthogroups
4. merge OG assignments -> augmented orthology (sandbox-tested glue)
5. rebuild MSA cache (CPU)
6. **retrain transformer scoring all clades (GPU)**  -> af_torch_preds_aug2.npz
7. extract DEG protein sequences
8. **ESM-2 embeddings -> protein families (GPU)**
9. save the 4 artifacts to download / push back

Bring back to the sandbox: `af_msa_cache_aug2.npz`, `af_torch_preds_aug2.npz`,
`og_assignments.csv`, `protein_families.csv` (+ optional `esm_embeddings.npz`).

In [ ]:
# 0. environment check
import torch, subprocess
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'select a GPU runtime (Runtime > Change runtime type)'

In [ ]:
# 1. repo + deps
import os
BRANCH = 'claude/vectorize-gex-propensity-NRqBW'
if not os.path.exists('/content/cell'):
    # set GH_TOKEN in Colab secrets for a private clone, or upload the repo
    tok = os.environ.get('GH_TOKEN', '')
    url = f'https://{tok}@github.com/nikku03/cell.git' if tok else 'https://github.com/nikku03/cell.git'
    !git clone --branch $BRANCH --depth 1 $url /content/cell
os.chdir('/content/cell')
!pip -q install transformers >/dev/null 2>&1
# mmseqs static binary (no conda)
if subprocess.run(['which','mmseqs'],capture_output=True).returncode != 0:
    !wget -q https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz && tar xzf mmseqs-linux-avx2.tar.gz
    os.environ['PATH'] = '/content/cell/mmseqs/bin:' + os.environ['PATH']
print('mmseqs:', subprocess.run(['which','mmseqs'],capture_output=True,text=True).stdout.strip() or 'MISSING')

In [ ]:
# 2. fetch DEG proteomes (NCBI). targets the 5 labelled DEG cells.
!python colab/fetch_proteomes.py --mode deg --all
!ls -la colab/work/proteomes | head

In [ ]:
# 3. assign DEG proteins to our OG space with mmseqs
!python colab/assign_ogs.py --proteomes colab/work/proteomes \
    --reps data/gtdb/og_reps.faa --out colab/work/og_assignments.csv
import pandas as pd; print(pd.read_csv('colab/work/og_assignments.csv').organism.value_counts())

In [ ]:
# 4. merge OG assignments -> augmented orthology (the sandbox-tested glue)
!python colab/merge_deg_ogs.py --assignments colab/work/og_assignments.csv

In [ ]:
# 5. rebuild MSA cache including the DEG cells (CPU, ~5 min)
!python colab/build_cache.py --labels_dir data/drive_import/labels_aug \
    --out outputs/orphan/af_msa_cache_aug2.npz --include_orphans

In [ ]:
# 6. RETRAIN transformer, scoring ALL clades incl. the new DEG ones (GPU)
!python colab/af_torch2.py --cache outputs/orphan/af_msa_cache_aug2.npz \
    --all_clades --tag _aug2
# (falls back to: python colab/af_torch.py --big --cache <...> if af_torch2 flags differ)

In [ ]:
# 7-8. DEG proteins -> ESM-2 embeddings -> protein families (GPU)
# adapter: per-org proteomes -> combined fasta + index the embedder expects
!python colab/deg_esm_prep.py --proteomes colab/work/proteomes \
    --fasta colab/work/deg_proteins.fasta --index colab/work/deg_protein_index.json
!python colab/embed_proteins_esm.py --fasta colab/work/deg_proteins.fasta \
    --index colab/work/deg_protein_index.json --out outputs/orphan/esm_embeddings.npy
# family clustering (kNN gap-slot match) is finalised sandbox-side from the .npy + index

In [ ]:
# 9. bundle the artifacts to bring back to the sandbox
import shutil, os
keep = ['outputs/orphan/af_msa_cache_aug2.npz',
        'outputs/orphan/af_torch_preds_aug2.npz',
        'colab/work/og_assignments.csv', 'colab/work/deg_protein_index.json',
        'outputs/orphan/esm_embeddings.npy']
os.makedirs('/content/deg_artifacts', exist_ok=True)
for f in keep:
    if os.path.exists(f): shutil.copy(f, '/content/deg_artifacts/')
shutil.make_archive('/content/deg_artifacts','zip','/content/deg_artifacts')
print('bundle ready: /content/deg_artifacts.zip'); print(os.listdir('/content/deg_artifacts'))
# from google.colab import files; files.download('/content/deg_artifacts.zip')
# OR push back:
# !git add outputs/orphan/af_*_aug2.npz colab/work/og_assignments.csv && \
#   git commit -m 'DEG run artifacts' && git push